# Lab 3 — обучение MLP на CIFAR-10 в Google Colab

1. **Runtime → Change runtime type → GPU** (T4 и т.п.).
2. Положите на Google Drive:
   - распакованный Kaggle-датасет в папку `cifar-10` (`trainLabels.csv` + `train/train/*.png`);
   - архив проекта `CV-lab3-project.zip` (соберите на ПК: `scripts/pack_for_colab.ps1`) **или** укажите `GIT_REPO_URL` ниже.
3. Запускайте ячейки по порядку. После каждого варианта чекпоинты копируются на Drive.

MLflow в Colab не используем (`--no-mlflow`). Отчёт соберите локально: `uv run python scripts/generate_report.py` после скачивания `checkpoints/`.

In [ ]:
# --- Настройки (измените под себя) ---
DRIVE_ROOT = "/content/drive/MyDrive/CV-lab3"  # папка на Drive
CIFAR_ON_DRIVE = f"{DRIVE_ROOT}/cifar-10"       # trainLabels.csv + train/...
PROJECT_ZIP = f"{DRIVE_ROOT}/CV-lab3-project.zip"  # если нет git
GIT_REPO_URL = ""  # например https://github.com/you/CV.git — если пусто, берём zip
GIT_BRANCH = "main"
WORK_DIR = "/content/CV-lab3"  # рабочая копия кода в Colab
DRIVE_BACKUP = f"{DRIVE_ROOT}/colab_outputs"  # бэкап чекпоинтов и кривых

# Для проверки пайплайна: 2 эпохи. Для сдачи лабы: None (из configs/base.yaml, 80)
EPOCHS_OVERRIDE = None
RUN_DEMO_PREPROCESSING = False

In [ ]:
import subprocess
import sys

def pip(*args: str) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

pip("google-colab")  # no-op вне Colab
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import shutil
import subprocess
import zipfile
from pathlib import Path

work = Path(WORK_DIR)
if work.exists():
    shutil.rmtree(work)
work.mkdir(parents=True)

if GIT_REPO_URL.strip():
    subprocess.check_call(
        ["git", "clone", "--depth", "1", "-b", GIT_BRANCH, GIT_REPO_URL.strip(), str(work)]
    )
else:
    zp = Path(PROJECT_ZIP)
    if not zp.is_file():
        raise FileNotFoundError(
            f"Нет {zp}. Загрузите zip на Drive или задайте GIT_REPO_URL.\n"
            f"На Windows: .\\scripts\\pack_for_colab.ps1"
        )
    with zipfile.ZipFile(zp) as zf:
        zf.extractall(work)
    if not (work / "pyproject.toml").is_file():
        raise RuntimeError(f"В {zp} нет pyproject.toml в корне архива")

os.chdir(work)
print("Project:", work.resolve())

In [ ]:
import torch
from pathlib import Path

cifar = Path(CIFAR_ON_DRIVE)
labels = cifar / "trainLabels.csv"
train_nested = cifar / "train" / "train"
train_flat = cifar / "train"
ok = labels.is_file() and (
    (train_nested.is_dir() and any(train_nested.glob("*.png")))
    or (train_flat.is_dir() and any(train_flat.glob("*.png")))
)
if not ok:
    raise FileNotFoundError(
        f"На Drive нет CIFAR-10: нужны {labels} и PNG в train/train или train.\n"
        f"Загрузите папку cifar-10 в {DRIVE_ROOT}/"
    )

os.environ["CIFAR10_DATA_ROOT"] = str(cifar.resolve())
print("CIFAR10_DATA_ROOT =", os.environ["CIFAR10_DATA_ROOT"])
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

In [ ]:
pip("-e", ".")

In [ ]:
import shutil
from pathlib import Path
from datetime import datetime

VARIANTS = [
    "configs/aug_variant_2.yaml",
    "configs/aug_variant_6.yaml",
    "configs/aug_variant_7.yaml",
]

def backup_run(tag: str) -> None:
    dest = Path(DRIVE_BACKUP) / tag
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True, exist_ok=True)
    for name in ("checkpoints", "artifacts"):
        src = Path(name)
        if src.is_dir():
            shutil.copytree(src, dest / name)
    print("Backed up to", dest)

stamp = datetime.utcnow().strftime("%Y%m%d_%H%M%S")

for cfg in VARIANTS:
    cmd = [
        sys.executable,
        "scripts/train.py",
        "--config",
        cfg,
        "--no-mlflow",
        "--download-test",
    ]
    if EPOCHS_OVERRIDE is not None:
        cmd.extend(["--epochs", str(EPOCHS_OVERRIDE)])
    print("\n>>>", " ".join(cmd))
    subprocess.check_call(cmd, cwd=WORK_DIR)
    tag = Path(cfg).stem + "_" + stamp
    backup_run(tag)

print("\nВсе прогоны завершены.")

In [ ]:
if RUN_DEMO_PREPROCESSING:
    subprocess.check_call([sys.executable, "scripts/demo_preprocessing.py"], cwd=WORK_DIR)
    backup_run("preprocess_demo")

In [ ]:
import zipfile
from pathlib import Path

out_zip = Path(DRIVE_BACKUP) / f"lab3_download_{stamp}.zip"
with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder in ("checkpoints", "artifacts"):
        root = Path(WORK_DIR) / folder
        if not root.is_dir():
            continue
        for path in root.rglob("*"):
            if path.is_file():
                zf.write(path, path.relative_to(WORK_DIR))
print("Скачайте с Drive:", out_zip)
print("Локально распакуйте в корень репозитория и выполните: uv run python scripts/generate_report.py")